# Prefill-Decode Disaggregation

**Prerequisites:** Python basics and the relevant model-serving concepts.  **Estimated time:** 35 minutes.

## Objectives

- Simulate a two-stage prefill/decode pipeline and inspect request latency.
- Explain at least one failure boundary and a diagnostic.
- Run deterministic assertions against a small synthetic example.


## Model and assumptions

Synthetic service times expose queue coupling; they do not predict hardware latency. All quantities below are deliberately small, deterministic teaching values. The paired local implementations are dependency-light (the MoE example uses NumPy) and do not call a model or network service. A synthetic proxy establishes accounting behavior, not production quality or speed.


In [ ]:
# Load the paired standalone implementation from the repository root.
import runpy
impl = runpy.run_path('modern-ai/implementations/33-prefill-decode-disaggregation.py')


## Transparent baseline

Inspect the short implementation above: the critical state and accounting are visible, so you can change one assumption at a time. In a real system, replace the toy inputs with representative traces and validate quality as well as resource use.


In [ ]:
requests=[(0,5,1),(1,1,4),(2,2,2),(3,1,1)]
rows=impl['simulate'](requests); lat=[x['latency'] for x in rows]
print(rows,'p50',impl['percentile'](lat,.5),'p95',impl['percentile'](lat,.95))
assert all(x['finish']>=x['first_token'] for x in rows)


## Comparison and metric

Use the printed counts, mask density, error, latency distribution, or decision labels to compare the alternatives. The selected metric is a proxy for a specific resource or behavior; it is not a substitute for end-to-end evaluation.

In [ ]:
# Edge case: reject invalid inputs or expose degraded behavior instead of hiding it.
try:
    impl['percentile']([], .95)
except (ValueError, MemoryError, IndexError) as exc:
    print(type(exc).__name__, exc)


## Failure cases and exercise

1. Change the topic parameter that exercises a boundary. Predict which metric changes before running the cell.
2. Identify a case where the proxy gives a misleading decision. State what real trace or quality label you would collect before deployment.

**Exercise:** Use the invariant below as a starting test, then add a second case that probes a boundary relevant to your workload.

In [ ]:
r=impl['simulate']([(0,2,1)])
assert r[0]['first_token'] < r[0]['finish']


## Summary and next lessons

This notebook made one prefill-decode disaggregation mechanism inspectable with deterministic accounting. Revisit the concept note at `../concepts/33-prefill-decode-disaggregation.md`, then compare with the neighboring serving lessons and evaluate on a representative workload before making a system choice.
